In [ ]:
!pip install fastapi uvicorn nest_asyncio pyngrok langchain openai

In [ ]:
from google.colab import userdata
userdata.get('smart_api_key')

'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'

In [ ]:
import sqlite3

# Connect to SQLite (creates file if it doesn't exist)
conn = sqlite3.connect("office.db")
cursor = conn.cursor()

# ----------------------------
# Table 1: Employees
# ----------------------------
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    role TEXT
)
""")

# ----------------------------
# Table 2: Meetings
# ----------------------------
cursor.execute("""
CREATE TABLE IF NOT EXISTS meetings(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    time TEXT,
    employee_id INTEGER,
    FOREIGN KEY(employee_id) REFERENCES employees(id)
)
""")

# ----------------------------
# Table 3: Tasks
# ----------------------------
cursor.execute("""
CREATE TABLE IF NOT EXISTS tasks(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    task TEXT NOT NULL,
    status TEXT DEFAULT 'pending',
    employee_id INTEGER,
    FOREIGN KEY(employee_id) REFERENCES employees(id)
)
""")

# ----------------------------
# Table 4: Documents
# ----------------------------
cursor.execute("""
CREATE TABLE IF NOT EXISTS documents(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    content TEXT,
    employee_id INTEGER,
    FOREIGN KEY(employee_id) REFERENCES employees(id)
)
""")

# Commit changes
conn.commit()
print("Database and tables created successfully!")

# Close connection
conn.close()

Database and tables created successfully!


In [ ]:
conn = sqlite3.connect("office.db")
cursor = conn.cursor()

# Sample Employees
cursor.execute("INSERT OR IGNORE INTO employees(name,email,role) VALUES (?,?,?)",
               ("Alice Hussain","alice@company.com","Manager"))
cursor.execute("INSERT OR IGNORE INTO employees(name,email,role) VALUES (?,?,?)",
               ("Bob Khan","bob@company.com","Employee"))

# Sample Meetings
cursor.execute("INSERT OR IGNORE INTO meetings(title,time,employee_id) VALUES (?,?,?)",
               ("Team Sync","10:00 AM",1))
cursor.execute("INSERT OR IGNORE INTO meetings(title,time,employee_id) VALUES (?,?,?)",
               ("Project Review","2:00 PM",2))

# Sample Tasks
cursor.execute("INSERT OR IGNORE INTO tasks(task,status,employee_id) VALUES (?,?,?)",
               ("Submit Report","pending",1))
cursor.execute("INSERT OR IGNORE INTO tasks(task,status,employee_id) VALUES (?,?,?)",
               ("Prepare Slides","pending",2))

# Sample Documents
cursor.execute("INSERT OR IGNORE INTO documents(title,content,employee_id) VALUES (?,?,?)",
               ("Leave Policy","All employees are entitled to 20 leaves per year.",1))
cursor.execute("INSERT OR IGNORE INTO documents(title,content,employee_id) VALUES (?,?,?)",
               ("Project Guidelines","Follow the Agile methodology for project execution.",2))

conn.commit()
conn.close()

print("Sample data inserted successfully!")

Sample data inserted successfully!


In [ ]:
conn = sqlite3.connect("office.db")
cursor = conn.cursor()

# Get all employees
cursor.execute("SELECT * FROM employees")
print("Employees:", cursor.fetchall())

# Get all meetings
cursor.execute("SELECT * FROM meetings")
print("Meetings:", cursor.fetchall())

# Get all tasks
cursor.execute("SELECT * FROM tasks")
print("Tasks:", cursor.fetchall())

# Get all documents
cursor.execute("SELECT * FROM documents")
print("Documents:", cursor.fetchall())

conn.close()

Employees: [(1, 'Alice Hussain', 'alice@company.com', 'Manager'), (2, 'Bob Khan', 'bob@company.com', 'Employee')]
Meetings: [(1, 'Team Sync', '10:00 AM', 1), (2, 'Project Review', '2:00 PM', 2)]
Tasks: [(1, 'Submit Report', 'pending', 1), (2, 'Prepare Slides', 'pending', 2)]
Documents: [(1, 'Leave Policy', 'All employees are entitled to 20 leaves per year.', 1), (2, 'Project Guidelines', 'Follow the Agile methodology for project execution.', 2)]


In [ ]:
conn = sqlite3.connect("office.db")
cursor = conn.cursor()

# Get all employees
cursor.execute("SELECT * FROM employees")
print("Employees:", cursor.fetchall())

# Get all meetings
cursor.execute("SELECT * FROM meetings")
print("Meetings:", cursor.fetchall())

# Get all tasks
cursor.execute("SELECT * FROM tasks")
print("Tasks:", cursor.fetchall())

# Get all documents
cursor.execute("SELECT * FROM documents")
print("Documents:", cursor.fetchall())

conn.close()

Employees: [(1, 'Alice Hussain', 'alice@company.com', 'Manager'), (2, 'Bob Khan', 'bob@company.com', 'Employee')]
Meetings: [(1, 'Team Sync', '10:00 AM', 1), (2, 'Project Review', '2:00 PM', 2)]
Tasks: [(1, 'Submit Report', 'pending', 1), (2, 'Prepare Slides', 'pending', 2)]
Documents: [(1, 'Leave Policy', 'All employees are entitled to 20 leaves per year.', 1), (2, 'Project Guidelines', 'Follow the Agile methodology for project execution.', 2)]


In [ ]:
%%writefile agent.py
from langchain.agents import initialize_agent
from langchain.tools import Tool
from langchain.llms import OpenAI
import sqlite3
import os

os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"  # add your key

# Connect database
conn = sqlite3.connect("office.db")
cursor = conn.cursor()

# Tool functions
def get_meetings():
    cursor.execute("SELECT title,time FROM meetings")
    data = cursor.fetchall()
    return str(data)

def get_tasks():
    cursor.execute("SELECT task,status FROM tasks")
    data = cursor.fetchall()
    return str(data)

def get_documents():
    cursor.execute("SELECT title,content FROM documents")
    data = cursor.fetchall()
    return str(data)

# Define tools
tools = [
    Tool(name="Meeting Checker", func=get_meetings, description="Check today's meetings"),
    Tool(name="Task Manager", func=get_tasks, description="Check pending tasks"),
    Tool(name="Document Finder", func=get_documents, description="Retrieve office documents")
]

# Initialize AI agent
llm = OpenAI(temperature=0)
agent = initialize_agent(tools, llm, agent="zero-shot-react-description", verbose=True)

Writing agent.py


In [ ]:
%%writefile main.py
from fastapi import FastAPI
from agent import agent

app = FastAPI()

@app.get("/")
def home():
    return {"message":"Smart Office Assistant Running"}

@app.get("/ask")
def ask(question:str):
    response = agent.run(question)
    return {"response":response}

Writing main.py


In [ ]:
!pip install flask flask-cors
!npm install -g localtunnel

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_18.x | bash -
!apt-get install -y nodejs
!npx create-react-app identity-protector --template cra-template

In [ ]:
# Copy the JSX file into the app
!cp /content/digital_identity_protector.jsx /content/identity-protector/src/App.js

In [ ]:
%cd /content/identity-protector
!npm run build
!npm install -g serve
!serve -s build -l 3000 &

In [ ]:
!npx localtunnel --port 3000